# Data EDA — listen-wiseer

Loads directly from `data/listen_wiseer.db` via DuckDB. Covers:
1. Schema overview & row counts
2. Null / coverage audit
3. Audio feature distributions
4. Genre taxonomy coverage
5. Temporal analysis (year / decade)
6. Playlist composition
7. Key & mode distribution
8. Faves analysis
9. Feature correlations
10. Audio profiles by genre group

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import warnings

warnings.filterwarnings("ignore")

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

DB = str(ROOT / "data/listen_wiseer.db")
con = duckdb.connect(DB, read_only=True)

print("Connected:", DB)

## 1. Schema overview

In [ ]:
tables = con.execute("SHOW TABLES").df()
print(tables.to_string(index=False))

In [ ]:
counts = {}
for tbl in tables["name"]:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    counts[tbl] = n

pd.Series(counts, name="row_count").sort_values(ascending=False)

In [ ]:
# track_profile view columns
con.execute("DESCRIBE track_profile").df()

## 2. Null / coverage audit

In [ ]:
tp = con.execute("SELECT * FROM track_profile").df()
print(f"track_profile rows: {len(tp):,}")

null_pct = (tp.isnull().sum() / len(tp) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(null_pct) * 0.35)))
null_pct.plot.barh(ax=ax)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Null % by column (track_profile)")
ax.set_xlabel("% null")
plt.tight_layout()

In [ ]:
# Genre taxonomy coverage
genre_coverage = con.execute("""
    SELECT
        COUNT(*) AS total_tracks,
        COUNT(first_genre) AS has_first_genre,
        COUNT(gen_4)       AS has_gen_4,
        COUNT(gen_6)       AS has_gen_6,
        COUNT(gen_8)       AS has_gen_8,
        COUNT(my_genre)    AS has_my_genre,
        COUNT(sub_genre)   AS has_sub_genre
    FROM track_profile
""").df()
genre_coverage

In [ ]:
# Tracks missing audio features
missing_af = con.execute("""
    SELECT COUNT(*) AS tracks_missing_audio_features
    FROM tracks t
    LEFT JOIN audio_features af USING (track_id)
    WHERE af.track_id IS NULL
""").fetchone()[0]
print(f"Tracks missing audio features: {missing_af:,}")

## 3. Audio feature distributions

In [ ]:
audio_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.flatten(), audio_cols, strict=False):
    data = tp[col].dropna()
    sns.histplot(data, kde=True, ax=ax, bins=40)
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Audio feature distributions", fontsize=14, y=1.01)
plt.tight_layout()

In [ ]:
tp[audio_cols + ["popularity"]].describe().round(3)

## 5. Temporal analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# year histogram
year_data = tp["year"].dropna().astype(int)
sns.histplot(year_data, bins=30, ax=axes[0])
axes[0].set_title("Tracks by release year")

# decade bar chart
decade_order = [f"{d}s" for d in range(1950, 2030, 10)]
decade_counts = (
    tp["decade"]
    .value_counts()
    .reindex([d for d in decade_order if d in tp["decade"].values], fill_value=0)
)
decade_counts.plot.bar(ax=axes[1], rot=45)
axes[1].set_title("Tracks by decade")

plt.tight_layout()

In [ ]:
# median audio features by decade
decade_audio = (
    tp.dropna(subset=["decade"])
    .groupby("decade")[audio_cols]
    .median()
    .reindex([d for d in decade_order if d in tp["decade"].values])
)
decade_audio.style.background_gradient(cmap="coolwarm", axis=0)

## 7. Key & mode distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# key distribution
key_counts = tp["key_labels"].value_counts()
key_counts.plot.bar(ax=axes[0], rot=45)
axes[0].set_title("Tracks by musical key")

# major vs minor
mode_counts = tp["mode_labels"].value_counts()
mode_counts.plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Major vs Minor")

plt.tight_layout()

In [ ]:
# top key_mode combos
tp["key_mode"].value_counts().head(15).to_frame("count")

## 8. Faves analysis

In [ ]:
faves_df = con.execute("""
    SELECT tp.*, f.score AS fave_score_raw
    FROM track_profile tp
    JOIN faves f USING (track_id)
    ORDER BY f.score DESC
""").df()
print(f"Faved tracks: {len(faves_df):,}")
faves_df["fave_score_raw"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(faves_df["fave_score_raw"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Fave score distribution")

# faves by genre group
faves_by_genre = faves_df.groupby("gen_4")["fave_score_raw"].mean().sort_values()
faves_by_genre.plot.barh(ax=axes[1])
axes[1].set_title("Avg fave score by gen_4")

plt.tight_layout()

In [ ]:
# fave score vs audio features (scatter with regression)
features_vs_fave = ["danceability", "energy", "valence", "acousticness", "instrumentalness"]
fig, axes = plt.subplots(1, len(features_vs_fave), figsize=(16, 3))
for ax, feat in zip(axes, features_vs_fave, strict=False):
    sns.regplot(
        data=faves_df,
        x=feat,
        y="fave_score_raw",
        scatter_kws={"alpha": 0.3, "s": 10},
        line_kws={"color": "red"},
        ax=ax,
    )
    ax.set_title(feat)
fig.suptitle("Fave score vs audio features", y=1.02)
plt.tight_layout()

## 9. Feature correlations

In [ ]:
corr_cols = audio_cols + ["popularity", "fave_score"]
corr = tp[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Audio feature correlation matrix")
plt.tight_layout()

In [ ]:
con.close()
print("Done.")